## _Spoticore Stage 2 PyTorchified_

We are going to refactor the stage 2 code to (kinda) match PyTorch's public API for building neural networks similar to spoticore.


### Imports

In [ ]:
from typing import Final
import torch
import torch.nn.functional as F
from math import sqrt
from collections.abc import Iterator
from typing import Protocol
from reader import read_all_unique_words

### Constants

In [2]:
SEED: Final[int] = 534150593
N_EMBD: Final[int] = 10
N_HIDDEN: Final[int] = 100

BLOCK_SIZE: Final[int] = 3

### Random seed generator

In [3]:
g = torch.Generator().manual_seed(SEED)

### Neural network architecture

In [4]:
class Layer(Protocol):
    def __init__(self) -> None: ...
    def __call__(self, x: torch.Tensor) -> torch.Tensor: ...
    def parameters(self) -> Iterator[torch.Tensor]: ...

In [5]:
class Linear:
    def __init__(self, fan_in: int, fan_out: int, bias: bool = True) -> None:
        # Kaiming Init with a gain of 1 for linear layer.
        self.weight = torch.randn((fan_in, fan_out), generator=g) / sqrt(fan_in)
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.bias

    def parameters(self) -> Iterator[torch.Tensor]:
        yield self.weight
        if self.bias is not None:
            yield self.bias

In [6]:
class BatchNorm1d:
    def __init__(
        self, num_features: int, eps: float = 1e-5, momentum: float = 0.1
    ) -> None:
        self.eps = eps
        self.momentum = momentum
        self.training = True
        # bn gain
        self.gamma = torch.ones(num_features)
        # bn bias
        self.beta = torch.zeros(num_features)
        # bn running mean and variance
        self.running_mean = torch.zeros(num_features)
        self.running_var = torch.ones(num_features)

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        # forward pass: calculate activations
        if self.training:
            xmean = x.mean(0, keepdim=True)
            xvar = x.var(0, keepdim=True)
        else:
            xmean = self.running_mean
            xvar = self.running_var

        # normalize to unit variance
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)

        self.out = self.gamma * xhat + self.beta

        # update running mean and var
        if self.training:
            with torch.no_grad():
                self.running_mean = (
                    1 - self.momentum
                ) * self.running_mean + self.momentum * xmean
                self.running_var = (
                    1 - self.momentum
                ) * self.running_var + self.momentum * xvar

        return self.out

    def parameters(self) -> Iterator[torch.Tensor]:
        yield self.gamma
        yield self.beta

In [7]:
class Tanh:
    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        self.out = torch.tanh(x)
        return self.out

    def parameters(self) -> Iterator[torch.Tensor]:
        return iter(())

### Build character-index mappings from training data

In [18]:
type StoiMap = dict[str, int]
type ItosMap = dict[int, str]


def build_vocab_mappings() -> tuple[list[str], StoiMap, ItosMap]:
    words = read_all_unique_words()

    chars = sorted(set("".join(words)))

    # create mappings with special token "." at index 0.
    stoi = {char: i + 1 for i, char in enumerate(chars)}
    stoi["."] = 0

    itos = {i: char for char, i in stoi.items()}

    return words, stoi, itos

### Construct model inputs from training data

In [9]:
words, stoi, itos = build_vocab_mappings()
vocab_size = len(stoi)

In [10]:
def construct_model_inputs(
    words: list[str], stoi: StoiMap, block_size: int = 3
) -> tuple[torch.Tensor, torch.Tensor]:
    block_size = 3

    X, Y = [], []

    for word in words:
        prev_chars_is = [0] * block_size
        for char in word + ".":
            nxt_char_i = stoi[char]
            X.append(prev_chars_is)
            Y.append(nxt_char_i)
            prev_chars_is = prev_chars_is[1:] + [nxt_char_i]

    return torch.tensor(X), torch.tensor(Y)

### Construct embedding lookup matrix

In [11]:
# Char -> N_EMBD row vector
C = torch.randn((vocab_size, N_EMBD), generator=g)

### Create deep MLP

In [12]:
layers: list[Layer] = [
    Linear(N_EMBD * BLOCK_SIZE, N_HIDDEN),
    Tanh(),
    Linear(N_HIDDEN, N_HIDDEN),
    Tanh(),
    Linear(N_HIDDEN, N_HIDDEN),
    Tanh(),
    Linear(N_HIDDEN, N_HIDDEN),
    Tanh(),
    Linear(N_HIDDEN, N_HIDDEN),
    Tanh(),
    Linear(N_HIDDEN, vocab_size),
]

In [13]:
import torch.nn.init as Init

gain = Init.calculate_gain("tanh")

### Kaiming Init for tanh

In [14]:
# make last layer less confident
with torch.no_grad():
    layers[-1].weight *= 0.1
    # apply gain for all tanh pre-activations
    # output layer does not perform tanh
    for layer in layers[:-1]:
        if isinstance(layer, Linear):
            layer.weight *= gain

In [15]:
params = [C] + [p for layer in layers for p in layer.parameters()]
print("Total parameters: ", sum(p.nelement() for p in params))

# ensure all parameters take part in backprogation
for p in params:
    p.requires_grad = True

Total parameters:  47607


### Train the model

In [16]:
X, Y = construct_model_inputs(words, stoi)

In [ ]:
iterations = 200_000
batch_size = 32
lr = (0.2, 0.1)


losses = []

for i in range(iterations):
    # Construct mini batch of inputs
    chosen_idxs = torch.randint(0, X.shape[0], (batch_size,), generator=g)

    # Extract the batches.
    X_batch, Y_batch = X[chosen_idxs], Y[chosen_idxs]

    # Forward pass.
    emb = C[X_batch]
    x = emb.view((batch_size, N_EMBD * BLOCK_SIZE))

    for layer in layers:
        x = layer(x)

    loss = F.cross_entropy(x, Y_batch)

    # Backward pass.
    for layer in layers:
        layer.out.retain_grad()

    for p in params:
        p.grad = None

    loss.backward()

    # Update params
    lr = lr[0] if i < 10_000 else lr[1]

torch.Size([32, 3, 10])
